# 支付宝营销策略效果分析

In [188]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [189]:
data = pd.read_csv('effect_tb.csv', header = None, names = ['dt', 'user_id', 'label', 'dmp_id'])

In [190]:
data.head()

,dt,user_id,label,dmp_id
0,1,1,0,1
1,1,1000004,0,1
2,1,1000004,0,2
3,1,1000006,0,1
4,1,1000006,0,3


In [191]:
data.shape

(2645958, 4)

处理重复行

In [123]:
# data.drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)

In [193]:
#只保留用户第一次点击广告的记录
data = data.sort_values(by = ['dt']).drop_duplicates(subset = ['user_id'], keep = 'first', inplace = False)
data[data.duplicated(keep = False)]

,dt,user_id,label,dmp_id


In [194]:
data.drop(columns = 'dt', inplace = True)

In [195]:
data.nunique()

user_id    2410683
label            2
dmp_id           3
dtype: int64

In [196]:
data.shape

(2410683, 3)

In [127]:
# data[data.duplicated(keep = False)].sort_values(by = ['user_id'])

,user_id,label,dmp_id
8529,1027,0,1
1485546,1027,0,1
1579415,1471,0,1
127827,1471,0,1
404862,2468,0,1
...,...,...,...
1382121,6264633,0,1
1382245,6264940,0,1
2575140,6264940,0,1
1382306,6265082,0,3


In [128]:
# data.drop_duplicates(keep = 'first', inplace = True)

In [129]:
# data[data.duplicated(keep = False)] 

处理缺失值

In [197]:
data.isnull().sum() #检查每列的缺失值，无缺失值

user_id    0
label      0
dmp_id     0
dtype: int64

创建一个数据透视表, 查看每个广告策略的点击和未点击人数

In [198]:
data_pivot = data.pivot_table(index = 'dmp_id', columns = 'label', values = 'user_id', aggfunc = 'count', margins = True)
data_pivot                     # 行             # 列                 # 值                # 聚合方式        # all

label,0,1,All
dmp_id,,,
1,1772301,23390,1795691
2,319527,5542,325069
3,282114,7809,289923
All,2373942,36741,2410683


计算最小样本量

In [199]:
# 计算各组的广告点击率
data_pivot.columns = ['nocli', 'cli', 'people']

data_pivot['click_rate'] = data_pivot['cli'] / data_pivot['people']
data_pivot.index = ['group1', 'group2', 'group3', 'all']
data_pivot

,nocli,cli,people,click_rate
group1,1772301,23390,1795691,0.013026
group2,319527,5542,325069,0.017049
group3,282114,7809,289923,0.026935
all,2373942,36741,2410683,0.015241


In [200]:
# 最小样本量取决于alpha、belta、方差、mde
alpha, power, mde = 0.05, 0.8, 0.01

click_rate1 = data_pivot.loc['group1']['click_rate']
p_mean = (click_rate1*2 + mde) / 2

In [201]:
import math
from scipy import stats
# 最小样本量计算函数
def min_sample_num(alpha, power, mde, p):
    z_a = stats.norm.ppf(1-alpha/2)
    z_b = stats.norm.ppf(power)
    variance_term = 2 * p * (1-p)
    min_sample = pow(z_a + z_b, 2) * variance_term / pow(mde, 2)
    return math.ceil(min_sample) # 向上取整

In [202]:
min_sample = min_sample_num(alpha, power, mde, p_mean)
min_sample

2779

In [203]:
# 三个组别的样本量均远大于2707， 满足最小样本需求

In [204]:
# 保存清洗后的文件
data.to_csv('output.csv', index = False)

In [205]:
# 重新读取
data = pd.read_csv('output.csv')

In [206]:
pivot = data.pivot_table(index = 'dmp_id', columns = 'label', values = 'user_id', aggfunc = 'count', margins = 'all')
pivot['rate'] = pivot.iloc[:, 1] / pivot.iloc[:,2]
pivot

label,0,1,All,rate
dmp_id,,,,
1,1772301,23390,1795691,0.013026
2,319527,5542,325069,0.017049
3,282114,7809,289923,0.026935
All,2373942,36741,2410683,0.015241


可以看到策略一和策略二相较对照组都有提升， 策略一提升0.3个百分点，策略二提升1.3个百分点，直邮策略二满足要求

接下来进行假设检验，看策略二的提升是否显著

设对照组的点击率为p1， 策略二的点击率为p2

H0：p1 >= p2

H1：p1 < p2

In [207]:
# z统计量的计算
def hypothesis_test_z(n1, p1, n2, p2):
    p_mean = (n1*p1 + n2*p2) / (n1 +n2)
    se = np.sqrt(p_mean*(1-p_mean)*(1/n1 +1/n2))
    z_score = (p1-p2) / se
    return z_score

In [208]:
n1 = pivot.iloc[0,2]
n2 = pivot.iloc[2,2]
p1 = pivot.iloc[0,3]
p2 = pivot.iloc[2,3]

z_a = stats.norm.ppf(alpha) # 左侧检验
z_score = hypothesis_test_z(n1, p1, n2, p2)
print(z_a, z_score)

-1.6448536269514729 -57.247835038202616


In [209]:
print('在显著性水平为0.05下拒绝原假设， 策略二有显著提升' if z_score < z_a else '接受原假设， 策略二没有显著提升')

在显著性水平为0.05下拒绝原假设， 策略二有显著提升


In [210]:
# 用python自带的函数计算
import statsmodels.stats.proportion as sp
count = np.array([pivot.iloc[0,1], pivot.iloc[2,1]])

z_score, p_value = sp.proportions_ztest(count, [n1,n2],alternative = 'smaller')
print("=== 使用statsmodels进行Z检验 ===")
print(f"Z统计量: {z_score:.4f}")
print(f"P值: {p_value:.10f}")
print(f"是否显著: {p_value < 0.05}")

=== 使用statsmodels进行Z检验 ===
Z统计量: -57.2478
P值: 0.0000000000
是否显著: True


## 结论

综上所述，两种营销策略中，策略二对广告点击率有显著提升效果，且相较于对照组点击率提升了近一倍，因而在两组营销策略中应选择第二组进行推广。